Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


Load Dataset

In [3]:
file_path = "data/SLIIT_IT_Student_Data_Template_filled-updated.csv"   # change to your actual file
df = pd.read_csv(file_path)

print("First 5 rows:")
print(df.head())
print("\nColumns:")
print(df.columns)


First 5 rows:
   Gender District_Location Education_Stream  \
0    Male           Colombo       Bio Stream   
1  Female           Gampaha     Maths Stream   
2    Male             Kandy       Bio Stream   
3  Female             Galle     Maths Stream   
4    Male            Matara  Commerce Stream   

                                         Soft_Skills  \
0           Communication, Teamwork, Time Management   
1     Leadership, Problem Solving, Critical Thinking   
2            Public Speaking, Teamwork, Adaptability   
3  Communication, Presentation Skills, Emotional ...   
4   Time Management, Work Ethic, Attention to Detail   

                                           Key_Skils Current_semester  GPA  \
0  Java, Python, Spring Boot, MySQL, Git, RESTful...             1Y1S  1.0   
1  SQL, Python, Pandas, Power BI, Tableau, Excel ...             1Y2S  1.3   
2  HTML5, CSS3, JavaScript ES6+, React.js, Tailwi...             2Y1S  2.1   
3  Java, Spring Boot, PostgreSQL, REST APIs, Mav

Data Cleaning

In [4]:
# 3.1 Remove unwanted 'Unnamed' columns (from Excel exports)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# 3.2 Define features and target
target_col = "Experties_Preferred_Career_Field"
feature_cols = ["Soft_Skills", "Key_Skils", "Current_semester", "GPA"]

# 3.3 Drop rows with missing target
df = df.dropna(subset=[target_col])

# 3.4 Handle missing + types in feature columns
df["Soft_Skills"] = df["Soft_Skills"].fillna("")
df["Key_Skils"] = df["Key_Skils"].fillna("")

df["GPA"] = pd.to_numeric(df["GPA"], errors="coerce")
df = df.dropna(subset=["GPA"])

df["Current_semester"] = df["Current_semester"].astype(str)

print("\nRows after cleaning:", len(df))
print("\nDtypes:")
print(df[feature_cols + [target_col]].dtypes)



Rows after cleaning: 499

Dtypes:
Soft_Skills                          object
Key_Skils                            object
Current_semester                     object
GPA                                 float64
Experties_Preferred_Career_Field     object
dtype: object


Create X, y and Encode Labels

In [5]:
# We’ll convert the string career names into integers using LabelEncoder.
X = df[feature_cols].copy()
y = df[target_col].copy()

# Encode target labels to integers
label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)

print("\nOriginal labels:", sorted(y.unique()))
print("Encoded labels:", np.unique(y_encoded))



Original labels: ['AI / Machine Learning Engineer', 'Cybersecurity / Network Engineer', 'Data Science & Analytics', 'DevOps / Cloud Engineer', 'Frontend / Web Developer', 'IT / Technology Generalist', 'Mobile App Developer', 'Software Engineer / Backend', 'UI/UX Designer']
Encoded labels: [0 1 2 3 4 5 6 7 8]


Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain size:", X_train.shape[0])
print("Test size:", X_test.shape[0])



Train size: 399
Test size: 100


Preprocessing (Same Idea: TF-IDF + OneHot + Scaling)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("soft_tfidf", TfidfVectorizer(), "Soft_Skills"),
        ("key_tfidf", TfidfVectorizer(), "Key_Skils"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Current_semester"]),
        ("num", StandardScaler(), ["GPA"])
    ]
)


Define the MLP (Neural Network) Model

In [8]:
# You can tune hidden_layer_sizes, max_iter, etc.
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),  # two hidden layers: 100 and 50 neurons
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)


Build the Pipeline (Preprocess → MLP)

In [9]:
mlp_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("mlp", mlp_model)
])


Train the MLP Model

In [10]:
print("Training MLP model...")
mlp_clf.fit(X_train, y_train)
print("✅ MLP training completed.")


Training MLP model...
✅ MLP training completed.


Evaluate Accuracy

In [15]:
y_pred_mlp = mlp_clf.predict(X_test)

mlp_acc = accuracy_score(y_test, y_pred_mlp)
print(f"\nMLP Accuracy: {mlp_acc:.4f}")



MLP Accuracy: 0.8700


Detailed Metrics (with Original Labels)

In [12]:
y_test_labels = label_enc.inverse_transform(y_test)
y_pred_labels = label_enc.inverse_transform(y_pred_mlp)

print("\nMLP Classification Report:")
print(classification_report(y_test_labels, y_pred_labels))

print("\nMLP Confusion Matrix (encoded labels):")
print(confusion_matrix(y_test, y_pred_mlp))



MLP Classification Report:
                                  precision    recall  f1-score   support

  AI / Machine Learning Engineer       1.00      0.86      0.92         7
Cybersecurity / Network Engineer       1.00      1.00      1.00         3
        Data Science & Analytics       0.76      0.93      0.83        27
         DevOps / Cloud Engineer       0.91      0.87      0.89        23
        Frontend / Web Developer       0.87      1.00      0.93        13
      IT / Technology Generalist       1.00      0.17      0.29         6
            Mobile App Developer       1.00      1.00      1.00         7
     Software Engineer / Backend       0.80      0.67      0.73         6
                  UI/UX Designer       1.00      1.00      1.00         8

                        accuracy                           0.87       100
                       macro avg       0.93      0.83      0.84       100
                    weighted avg       0.88      0.87      0.86       100


MLP Co

Predict Career for a New Student

In [14]:
new_student = pd.DataFrame([{
    "Soft_Skills": "Communication, Teamwork, Problem solving",
    "Key_Skils": "Python, SQL, Web Development",
    "Current_semester": "2Y1S",
    "GPA": 3.25
}])

pred_encoded = mlp_clf.predict(new_student)[0]
pred_label = label_enc.inverse_transform([pred_encoded])[0]

print("\nPredicted Career Field (MLP):", pred_label)
print(f"\nMLP Accuracy: {mlp_acc:.4f}")




Predicted Career Field (MLP): Data Science & Analytics

MLP Accuracy: 0.8700
